# This notebook is for loading data and training the TF-IDF


## 1.Importing libraries

In [1]:
import pandas as pd
from pathlib import Path
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

from src.utils.text_cleaner import clean_text

[nltk_data] Downloading package stopwords to C:\Users\PHILLIPPA
[nltk_data]     SANYAMAHWE\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\PHILLIPPA
[nltk_data]     SANYAMAHWE\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 2. Load the CSV job dataset


In [2]:
df = pd.read_csv("data/job_dataset.csv")

df.head()

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


## 3. Combining the useful columns
1. Title
2. Skills
3. Responsibilities
4. Keywords

In [3]:
df["combined_text"] = (
    df["Title"].fillna("") + " "+
    df["Skills"].fillna("").str.replace(";", " ") + " " +
    df["Responsibilities"].fillna("").str.replace(";", " ") + " " +
    df["Keywords"].fillna("").str.replace(";", " ")
)


In [4]:
df['combined_text'].head()

0    .NET Developer C#  VB.NET basics  .NET Framewo...
1    .NET Developer C#  .NET Framework basics  ASP....
2    .NET Developer C#  VB.NET basics  .NET Core  A...
3    .NET Developer C#  .NET Framework  ASP.NET bas...
4    .NET Developer C#  ASP.NET  MVC  Entity Framew...
Name: combined_text, dtype: str

In [5]:
csv_corpus = [clean_text(text) for text in df["combined_text"]]

len(csv_corpus)

1068

## 4. Loading the text files of job descriptions

In [6]:
txt_dir = Path("data/job_descriptions")

txt_corpus = []

for file in txt_dir.glob("*.txt"):
    text = file.read_text(encoding="utf-8")
    cleaned = clean_text(text)
    txt_corpus.append(cleaned)

len(txt_corpus)

23

## 5. Combining both datasets

In [7]:
full_job_corpus = csv_corpus + txt_corpus

## 6. Training the TF-IDF Vectorizer

In [9]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_df=0.8
)

tfidf_matrix = vectorizer.fit_transform(full_job_corpus)

tfidf_matrix.shape

(1091, 6889)

## 7. Inspecting what the model learned

In [10]:
vectorizer.get_feature_names_out()[:50]

array(['3d', '3d content', '3d design', '3d graphic', '3d model',
       '3d modeling', '3d modelinganimation', 'ab', 'ab testing',
       'ability', 'ability manage', 'ability work', 'academic',
       'academic personal', 'academic project', 'access', 'accessibility',
       'accessibility compliance', 'accessibility design',
       'accessibility standard', 'accessibility user', 'account',
       'accounting', 'accuracy', 'accuracy clarity',
       'accuracy collaborate', 'accuracy continuously', 'accuracy edit',
       'accuracy support', 'accuracy use', 'accurate', 'accurate record',
       'across', 'across channel', 'across department', 'across design',
       'across multiple', 'across platform', 'across product',
       'across project', 'across team', 'action', 'action iac',
       'action infrastructure', 'action monitoring', 'actionable',
       'actionable insight', 'actionable recommendation', 'active',
       'active directory'], dtype=object)

In [11]:
with open("tfidf_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)